# Missing Values (NaN) Analysis in Municipal Census Data
## Spain 1996-2024: Demographic Data Integrity

*Notebook to identify and characterize municipalities with incomplete data*

In [1]:
"""
Notebook: 05_nan_analysis_padron_municipal.ipynb
Author: Data Analysis Pipeline
Created during exploratory phase (Feb 2026)
Last updated: 2026-02-10


Purpose:
    Identify and analyze missing values (NaN) in municipal census data
    using cleaned historical population records (Padrón Municipal, 1996–2024).

Scope:
    - Input: Raw census data (01_padron_clean_1996_2024.csv)
    - Output: Missing data inventory and integrity reports
    - Spatial unit: Municipality (INE codes)
    - Temporal unit: Annual (1996-2024, excluding 1997)
    - Demographic categories: Men, Women, Total

Analysis focus:
    - Detection of missing values by municipality and category
    - Temporal patterns of data availability
    - Spatial distribution of missing data
    - Data integrity assessment and quality metrics
    - Identification of problematic municipalities and time periods
    - Preparation of data quality documentation for downstream analysis

Notes:
    - This notebook assumes cleaned and standardized input data
    - Results are descriptive and diagnostic, not inferential
    - Missing values are analyzed but not imputed at this stage
    - Output can inform data filtering decisions for temporal analysis
    - Geographic data is integrated for territorial characterization

Key outputs:
    1. 03_nan_analysis_detailed.csv - Long format (114 records)
    2. 03_nan_analysis_pivoted.csv - Wide format (38 municipalities)
    3. 03_nan_analysis_by_comarca.csv - County-level summary (33 counties)
    4. 03_nan_analysis.gpkg - GeoPackage for GIS integration

Status:
    - Core diagnostic workflow implemented
    - Quality metrics considered stable and reproducible
    - Notebook may be extended with additional stratification or filtering
    - Results serve as baseline for population variation analysis
"""

'\nNotebook: 05_nan_analysis_padron_municipal.ipynb\nAuthor: Data Analysis Pipeline\nCreated during exploratory phase (Feb 2026)\nLast updated: 2026-02-10\n\n\nPurpose:\n    Identify and analyze missing values (NaN) in municipal census data\n    using cleaned historical population records (Padrón Municipal, 1996–2024).\n\nScope:\n    - Input: Raw census data (01_padron_clean_1996_2024.csv)\n    - Output: Missing data inventory and integrity reports\n    - Spatial unit: Municipality (INE codes)\n    - Temporal unit: Annual (1996-2024, excluding 1997)\n    - Demographic categories: Men, Women, Total\n\nAnalysis focus:\n    - Detection of missing values by municipality and category\n    - Temporal patterns of data availability\n    - Spatial distribution of missing data\n    - Data integrity assessment and quality metrics\n    - Identification of problematic municipalities and time periods\n    - Preparation of data quality documentation for downstream analysis\n\nNotes:\n    - This noteb

## Notebook Metadata

| Field | Value |
|-------|-------|
| **Name** | 03_nan_analysis_padron_municipal.ipynb |
| **Purpose** | Identify and analyze missing values (NaN) in municipal census data |
| **Dataset** | 01_padron_clean_1996_2024.csv |
| **Geographic Data** | mun_geographic_administrative_hierarchy.csv |
| **Period** | 1996-2024 (28 years) |
| **Categories** | Men, Women, Total |
| **Municipalities** | 8,132 |
| **Creation Date** | 2026-02-10 |
| **Author** | Automated Analysis |

In [2]:
# Import required libraries
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## 1. Data Loading

In [3]:
# Load municipal census data
# Base data directory (portable across Windows, Linux, Codespaces)
DATA_DIR = Path(
    r"/workspaces/rural-migration-land-use-spain/data/demography/processed"
)

# File Path
fp_padron_clean = DATA_DIR / "01_padron_clean_1996_2024.csv"

# Read file - IMPORTANT: Load Mun_Code as string to preserve leading zeros
padron = pd.read_csv(fp_padron_clean, dtype={'Mun_Code': str})

# Ensure Mun_Code has 5 digits with leading zeros
padron['Mun_Code'] = padron['Mun_Code'].str.zfill(5)

print(f"✓ Census data loaded: {len(padron):,} records")
print(f"  Municipalities: {padron['Mun_Code'].nunique():,}")
print(f"  Mun_Code format: {padron['Mun_Code'].dtype} (e.g., {padron['Mun_Code'].iloc[0]})")
print(f"  Unique years: {sorted(padron['Year'].unique())}")
print(f"  Categories: {padron['Cat'].unique().tolist()}")
print(f"\nFirst records:")
padron.head(10)

✓ Census data loaded: 683,088 records
  Municipalities: 8,132
  Mun_Code format: str (e.g., 01001)
  Unique years: [np.int64(1996), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
  Categories: ['Hombres', 'Mujeres', 'Total']

First records:


,Mun_Code,Mun,Cat,Year,Pop
0,01001,Alegría-Dulantzi,Hombres,1996,640.0
1,01001,Alegría-Dulantzi,Mujeres,1996,594.0
2,01001,Alegría-Dulantzi,Total,1996,1234.0
3,01001,Alegría-Dulantzi,Hombres,1998,656.0
4,01001,Alegría-Dulantzi,Mujeres,1998,603.0
5,01001,Alegría-Dulantzi,Total,1998,1259.0
6,01001,Alegría-Dulantzi,Hombres,1999,696.0
7,01001,Alegría-Dulantzi,Mujeres,1999,633.0
8,01001,Alegría-Dulantzi,Total,1999,1329.0
9,01001,Alegría-Dulantzi,Hombres,2000,731.0


In [4]:
# Load geographic data
# Spatial data directory (portable across Windows, Linux, Codespaces)
SPATIAL_DIR = Path(
    r"/workspaces/rural-migration-land-use-spain/data/spatial/derived"
)

# File Path
fp_geo = SPATIAL_DIR / "mun_geographic_administrative_hierarchy.csv"

# Read file - IMPORTANT: Load Mun_Code as string to preserve leading zeros
geo = pd.read_csv(fp_geo, sep=';', dtype={'Mun_Code': str})
geo.columns = geo.columns.str.strip()

# Ensure Mun_Code has 5 digits with leading zeros
geo['Mun_Code'] = geo['Mun_Code'].str.zfill(5)

print(f"✓ Geographic data loaded: {len(geo):,} municipalities")
print(f"  Mun_Code format: {geo['Mun_Code'].dtype} (e.g., {geo['Mun_Code'].iloc[0]})")
print(f"  Counties (Comarcas): {geo['Comarca_Code'].nunique():.0f}")
print(f"  Provinces: {geo['Prov_Code'].nunique():.0f}")
print(f"  Autonomous Communities: {geo['CCAA_Code'].nunique():.0f}")
print(f"\nFirst records:")
geo.head()

✓ Geographic data loaded: 8,132 municipalities
  Mun_Code format: str (e.g., 09298)
  Counties (Comarcas): 349
  Provinces: 52
  Autonomous Communities: 19

First records:


,Mun_Code,Mun_Name,Comarca_Code,Comarca_Name,Prov_Code,Prov_Name,CCAA_Code,CCAA_Name
0,09298,Quintanilla San García,902.0,BUREBA-EBRO,9,Burgos,7,Castilla y León
1,09301,Quintanilla Vivar,908.0,ARLANZON,9,Burgos,7,Castilla y León
2,09302,Rabanera del Pinar,903.0,DEMANDA,9,Burgos,7,Castilla y León
3,05105,Hoyos del Espino,504.0,GREDOS,5,Ávila,7,Castilla y León
4,05106,Hoyos de Miguel Muñoz,504.0,GREDOS,5,Ávila,7,Castilla y León


## 2. Exploratory Analysis of Missing Values

In [5]:
# General analysis of NaN values
print("="*70)
print("GENERAL SUMMARY OF NaN")
print("="*70)

total_values = len(padron)
nan_count = padron['Pop'].isna().sum()
nan_pct = (nan_count / total_values) * 100

print(f"\nTotal records: {total_values:,}")
print(f"Missing values (NaN): {nan_count:,}")
print(f"Valid values: {total_values - nan_count:,}")
print(f"Percentage NaN: {nan_pct:.3f}%")

# NaN by category
print(f"\nMissing values by category:")
for cat in padron['Cat'].unique():
    subset = padron[padron['Cat'] == cat]
    nan_cat = subset['Pop'].isna().sum()
    pct_cat = (nan_cat / len(subset)) * 100
    print(f"  {cat:10s}: {nan_cat:5,} ({pct_cat:6.3f}%)")

GENERAL SUMMARY OF NaN

Total records: 683,088
Missing values (NaN): 1,467
Valid values: 681,621
Percentage NaN: 0.215%

Missing values by category:
  Hombres   :   489 ( 0.215%)
  Mujeres   :   489 ( 0.215%)
  Total     :   489 ( 0.215%)


In [6]:
# NaN by year (Total category only)
total_df = padron[padron['Cat'] == 'Total']

print("\nMissing values by year (Total category):")
print("-" * 50)

nan_by_year = []
for year in sorted(total_df['Year'].unique()):
    subset = total_df[total_df['Year'] == year]
    nan_count = subset['Pop'].isna().sum()
    nan_pct = (nan_count / len(subset)) * 100
    nan_by_year.append({
        'Year': int(year),
        'Municipalities_with_NaN': nan_count,
        'Pct_NaN': round(nan_pct, 3)
    })
    print(f"{int(year)}: {nan_count:3d} municipalities ({nan_pct:6.3f}%)")

nan_year_df = pd.DataFrame(nan_by_year)
print(f"\nYear with most NaN: {nan_year_df.loc[nan_year_df['Municipalities_with_NaN'].idxmax(), 'Year']:.0f} ({nan_year_df['Municipalities_with_NaN'].max()} municipalities)")
print(f"Year with least NaN: {nan_year_df.loc[nan_year_df['Municipalities_with_NaN'].idxmin(), 'Year']:.0f} ({nan_year_df['Municipalities_with_NaN'].min()} municipalities)")


Missing values by year (Total category):
--------------------------------------------------
1996:  38 municipalities ( 0.467%)
1998:  36 municipalities ( 0.443%)
1999:  33 municipalities ( 0.406%)
2000:  30 municipalities ( 0.369%)
2001:  27 municipalities ( 0.332%)
2002:  26 municipalities ( 0.320%)
2003:  26 municipalities ( 0.320%)
2004:  25 municipalities ( 0.307%)
2005:  25 municipalities ( 0.307%)
2006:  24 municipalities ( 0.295%)
2007:  23 municipalities ( 0.283%)
2008:  22 municipalities ( 0.271%)
2009:  22 municipalities ( 0.271%)
2010:  20 municipalities ( 0.246%)
2011:  18 municipalities ( 0.221%)
2012:  18 municipalities ( 0.221%)
2013:  17 municipalities ( 0.209%)
2014:  16 municipalities ( 0.197%)
2015:  14 municipalities ( 0.172%)
2016:   8 municipalities ( 0.098%)
2017:   8 municipalities ( 0.098%)
2018:   8 municipalities ( 0.098%)
2019:   1 municipalities ( 0.012%)
2020:   1 municipalities ( 0.012%)
2021:   1 municipalities ( 0.012%)
2022:   1 municipalities ( 0.012

## 3. Identification of Municipalities with Missing Data

In [7]:
# Create detailed analysis by municipality and category
print("Analyzing missing data by municipality and category...")

# Ensure all Mun_Code are strings with 5 digits
padron['Mun_Code'] = padron['Mun_Code'].astype(str).str.zfill(5)
geo['Mun_Code'] = geo['Mun_Code'].astype(str).str.zfill(5)

# Define the complete set of years (1996-2024 excluding 1997)
ALL_YEARS = set(range(1996, 2025)) - {1997}  # 28 years total
TOTAL_YEARS = len(ALL_YEARS)  # Should be 28

print(f"  Reference period: {min(ALL_YEARS)}-{max(ALL_YEARS)} (excluding 1997)")
print(f"  Total years in reference: {TOTAL_YEARS}")

# Build detailed analysis
results = []

# Get all municipalities that have at least one NaN
mun_with_nan = padron[padron['Pop'].isna()]['Mun_Code'].unique()

for mun_code in mun_with_nan:
    mun_data = padron[padron['Mun_Code'] == mun_code]
    mun_name = mun_data['Mun'].iloc[0]
    
    # Get geographic data
    geo_match = geo[geo['Mun_Code'] == mun_code]
    if len(geo_match) > 0:
        comarca_code = geo_match['Comarca_Code'].iloc[0]
        comarca_name = geo_match['Comarca_Name'].iloc[0]
        prov_code = geo_match['Prov_Code'].iloc[0]
        prov_name = geo_match['Prov_Name'].iloc[0]
        ccaa_code = geo_match['CCAA_Code'].iloc[0]
        ccaa_name = geo_match['CCAA_Name'].iloc[0]
    else:
        comarca_code = comarca_name = prov_code = prov_name = ccaa_code = ccaa_name = None
    
    # Analyze each category
    for cat in ['Hombres', 'Mujeres', 'Total']:
        cat_data = mun_data[mun_data['Cat'] == cat]
        
        # Years present in dataset for this municipality/category
        years_present = set(cat_data['Year'].unique())
        
        # Years with valid data (not NaN)
        years_with_data = set(cat_data[cat_data['Pop'].notna()]['Year'].unique())
        
        # Missing years = years in reference that don't have valid data
        # This includes: years not in dataset + years with NaN
        years_missing = ALL_YEARS - years_with_data
        
        # Only add if there are missing years
        if len(years_missing) > 0:
            results.append({
                'Mun_Code': mun_code,
                'Mun_Name': mun_name,
                'Comarca_Code': comarca_code,
                'Comarca_Name': comarca_name,
                'Prov_Code': prov_code,
                'Prov_Name': prov_name,
                'CCAA_Code': ccaa_code,
                'CCAA_Name': ccaa_name,
                'Category': cat,
                'Total_Years': TOTAL_YEARS,  # Always 28
                'Years_With_Data': len(years_with_data),
                'Years_Missing': len(years_missing),
                'Pct_Missing': round((len(years_missing) / TOTAL_YEARS) * 100, 2),
                'Missing_Years': ','.join(map(str, sorted(years_missing)))
            })

# Create DataFrame
df_detailed = pd.DataFrame(results)

# Sort by comarca, municipality, category
df_detailed = df_detailed.sort_values(
    ['Comarca_Name', 'Mun_Name', 'Category']
).reset_index(drop=True)

print(f"✓ Analysis completed")
print(f"\nMunicipalities with missing data: {df_detailed['Mun_Code'].nunique()}")
print(f"Total records (M+W+T): {len(df_detailed)}")

# Verify calculations are correct
assert (df_detailed['Years_With_Data'] + df_detailed['Years_Missing'] == TOTAL_YEARS).all(), \
    "ERROR: Years_With_Data + Years_Missing should equal Total_Years"
assert (df_detailed['Pct_Missing'] <= 100).all(), \
    "ERROR: Pct_Missing should never exceed 100%"
assert (df_detailed['Years_With_Data'] >= 0).all(), \
    "ERROR: Years_With_Data should never be negative"

print(f"\n✓ All validation checks passed")
print(f"  - Total_Years is always {TOTAL_YEARS}")
print(f"  - Years_With_Data + Years_Missing = {TOTAL_YEARS} for all records")
print(f"  - Pct_Missing range: {df_detailed['Pct_Missing'].min():.2f}% - {df_detailed['Pct_Missing'].max():.2f}%")

print(f"\nFirst 10 records:")
print(df_detailed.head(10).to_string())

Analyzing missing data by municipality and category...
  Reference period: 1996-2024 (excluding 1997)
  Total years in reference: 28
✓ Analysis completed

Municipalities with missing data: 38
Total records (M+W+T): 114

✓ All validation checks passed
  - Total_Years is always 28
  - Years_With_Data + Years_Missing = 28 for all records
  - Pct_Missing range: 3.57% - 96.43%

First 10 records:
  Mun_Code         Mun_Name  Comarca_Code  Comarca_Name  Prov_Code Prov_Name  CCAA_Code           CCAA_Name Category  Total_Years  Years_With_Data  Years_Missing  Pct_Missing                                                                                                  Missing_Years
0    06903         Guadiana         601.0  ALBURQUERQUE          6   Badajoz         11         Extremadura  Hombres           28               12             16        57.14                                1996,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012
1    06903         Guadiana       

## 4. Top Affected Municipalities

In [8]:
# Top 15 municipalities with highest % of NaN (using Total)
total_df = df_detailed[df_detailed['Category'] == 'Total']
top_affected = total_df.nlargest(15, 'Pct_Missing')[['Mun_Code', 'Mun_Name', 'Comarca_Name', 'Prov_Name', 'Years_Missing', 'Pct_Missing', 'Missing_Years']]

print("TOP 15 MOST AFFECTED MUNICIPALITIES")
print("="*100)
print(top_affected.to_string())

print(f"\n\nStatistics of affected municipalities:")
print(f"  Maximum % of NaN: {total_df['Pct_Missing'].max():.2f}%")
print(f"  Average % of NaN: {total_df['Pct_Missing'].mean():.2f}%")
print(f"  Median % of NaN: {total_df['Pct_Missing'].median():.2f}%")
print(f"  Minimum % of NaN: {total_df['Pct_Missing'].min():.2f}%")

TOP 15 MOST AFFECTED MUNICIPALITIES
    Mun_Code                   Mun_Name           Comarca_Name Prov_Name  Years_Missing  Pct_Missing                                                                                                                           Missing_Years
47     48916                   Usansolo            GRAN BILBAO   Bizkaia             27        96.43  1996,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
5      18077                     Fornes                 ALHAMA   Granada             22        78.57                           1996,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
14     21902       Zarza-Perrunal, La\t      ANDEVALO ORIENTAL    Huelva             22        78.57                           1996,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
26     14902    

## 5. Analysis by County (Comarca)

In [9]:
# Summary by county
comarca_summary = []

for comarca in sorted(df_detailed['Comarca_Name'].dropna().unique()):
    comarca_data = df_detailed[df_detailed['Comarca_Name'] == comarca]
    
    comarca_summary.append({
        'Comarca_Code': comarca_data['Comarca_Code'].iloc[0],
        'Comarca_Name': comarca,
        'Prov_Name': comarca_data['Prov_Name'].iloc[0],
        'CCAA_Name': comarca_data['CCAA_Name'].iloc[0],
        'Mun_Count': comarca_data['Mun_Code'].nunique(),
        'Avg_Pct_Missing': round(comarca_data[comarca_data['Category'] == 'Total']['Pct_Missing'].mean(), 2),
        'Max_Pct_Missing': comarca_data[comarca_data['Category'] == 'Total']['Pct_Missing'].max()
    })

df_comarca = pd.DataFrame(comarca_summary)
df_comarca = df_comarca.sort_values('Mun_Count', ascending=False)

print("SUMMARY BY COUNTY (sorted by number of affected municipalities)")
print("="*100)
print(df_comarca.to_string())

SUMMARY BY COUNTY (sorted by number of affected municipalities)
    Comarca_Code            Comarca_Name               Prov_Name             CCAA_Name  Mun_Count  Avg_Pct_Missing  Max_Pct_Missing
1         1807.0                  ALHAMA                 Granada             Andalucía          2            73.22            78.57
29        2902.0       SERRANIA DE RONDA                  Málaga             Andalucía          2            67.86            67.86
23        1006.0   NAVALMORAL DE LA MATA                 Cáceres           Extremadura          2            64.28            67.86
22        1303.0                  MANCHA             Ciudad Real    Castilla-La Mancha          2            10.71            10.71
17        1805.0                IZNALLOZ                 Granada             Andalucía          2            66.08            67.86
0          601.0            ALBURQUERQUE                 Badajoz           Extremadura          1            57.14            57.14
2         25

## 6. Pivoted Analysis (1 municipality = 1 row)

In [10]:
# Create pivoted format (one row per municipality)
pivot_data = []

for mun_code in df_detailed['Mun_Code'].unique():
    mun_rows = df_detailed[df_detailed['Mun_Code'] == mun_code]
    
    first_row = mun_rows.iloc[0]
    
    entry = {
        'Mun_Code': mun_code,
        'Mun_Name': first_row['Mun_Name'],
        'Comarca_Code': first_row['Comarca_Code'],
        'Comarca_Name': first_row['Comarca_Name'],
        'Prov_Code': first_row['Prov_Code'],
        'Prov_Name': first_row['Prov_Name'],
        'CCAA_Code': first_row['CCAA_Code'],
        'CCAA_Name': first_row['CCAA_Name'],
    }
    
    for cat in ['Hombres', 'Mujeres', 'Total']:
        cat_row = mun_rows[mun_rows['Category'] == cat]
        if len(cat_row) > 0:
            cr = cat_row.iloc[0]
            entry[f'{cat}_Total_Years'] = int(cr['Total_Years'])
            entry[f'{cat}_Years_With_Data'] = int(cr['Years_With_Data'])
            entry[f'{cat}_Years_Missing'] = int(cr['Years_Missing'])
            entry[f'{cat}_Pct_Missing'] = cr['Pct_Missing']
            entry[f'{cat}_Missing_Years'] = cr['Missing_Years']
    
    pivot_data.append(entry)

df_pivoted = pd.DataFrame(pivot_data)
df_pivoted = df_pivoted.sort_values(['Comarca_Name', 'Mun_Name']).reset_index(drop=True)

print(f"Municipalities with missing data (pivoted format): {len(df_pivoted)}")
print(f"Columns: {len(df_pivoted.columns)}")
print(f"\nFirst 5 municipalities:")
df_pivoted.head()

Municipalities with missing data (pivoted format): 38
Columns: 23

First 5 municipalities:


,Mun_Code,Mun_Name,Comarca_Code,Comarca_Name,Prov_Code,Prov_Name,CCAA_Code,CCAA_Name,Hombres_Total_Years,Hombres_Years_With_Data,Hombres_Years_Missing,Hombres_Pct_Missing,Hombres_Missing_Years,Mujeres_Total_Years,Mujeres_Years_With_Data,Mujeres_Years_Missing,Mujeres_Pct_Missing,Mujeres_Missing_Years,Total_Total_Years,Total_Years_With_Data,Total_Years_Missing,Total_Pct_Missing,Total_Missing_Years
0,06903,Guadiana,601.0,ALBURQUERQUE,6,Badajoz,11,Extremadura,28,12,16,57.14,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2...",28,12,16,57.14,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2...",28,12,16,57.14,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2..."
1,18077,Fornes,1807.0,ALHAMA,18,Granada,1,Andalucía,28,6,22,78.57,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2...",28,6,22,78.57,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2...",28,6,22,78.57,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2..."
2,18106,Játar,1807.0,ALHAMA,18,Granada,1,Andalucía,28,9,19,67.86,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2...",28,9,19,67.86,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2...",28,9,19,67.86,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2..."
3,25913,Riu de Cerdanya,2503.0,ALTO URGEL,25,Lleida,9,Cataluña/Catalunya,28,26,2,7.14,"1996,1998",28,26,2,7.14,"1996,1998",28,26,2,7.14,"1996,1998"
4,21902,"Zarza-Perrunal, La\t",2103.0,ANDEVALO ORIENTAL,21,Huelva,1,Andalucía,28,6,22,78.57,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2...",28,6,22,78.57,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2...",28,6,22,78.57,"1996,1998,1999,2000,2001,2002,2003,2004,2005,2..."


## 7. Final Statistics

In [11]:
# Final statistics
print("="*70)
print("FINAL STATISTICS")
print("="*70)

# Define reference period
ALL_YEARS = set(range(1996, 2025)) - {1997}  # 28 years total
TOTAL_YEARS = len(ALL_YEARS)

print(f"\nOriginal dataset:")
print(f"  Total municipalities: {padron['Mun_Code'].nunique():,}")
print(f"  Total records: {len(padron):,}")
nan_count = padron['Pop'].isna().sum()
nan_pct = (nan_count / len(padron)) * 100
print(f"  Missing values (NaN): {nan_count:,} ({nan_pct:.3f}%)")

print(f"\nReference period:")
print(f"  Years: {min(ALL_YEARS)}-{max(ALL_YEARS)} (excluding 1997)")
print(f"  Total years: {TOTAL_YEARS}")

print(f"\nAffected municipalities:")
affected = df_detailed['Mun_Code'].nunique()
total_mun = padron['Mun_Code'].nunique()
print(f"  With some missing data: {affected} ({(affected/total_mun*100):.2f}%)")
print(f"  Complete (no missing): {total_mun - affected} ({((total_mun-affected)/total_mun*100):.2f}%)")

print(f"\nData integrity (among affected municipalities):")
# Use df_detailed which has Category='Total' for overall stats
total_cat = df_detailed[df_detailed['Category'] == 'Total']
print(f"  Minimum % missing: {total_cat['Pct_Missing'].min():.2f}%")
print(f"  Maximum % missing: {total_cat['Pct_Missing'].max():.2f}%")
print(f"  Average % missing: {total_cat['Pct_Missing'].mean():.2f}%")
print(f"  Median % missing: {total_cat['Pct_Missing'].median():.2f}%")

print(f"\n✓ Validation:")
print(f"  - Total_Years is always {TOTAL_YEARS} for all records")
print(f"  - Years_With_Data + Years_Missing = {TOTAL_YEARS}")
print(f"  - Pct_Missing range: 0% - 100%")

FINAL STATISTICS

Original dataset:
  Total municipalities: 8,132
  Total records: 683,088
  Missing values (NaN): 1,467 (0.215%)

Reference period:
  Years: 1996-2024 (excluding 1997)
  Total years: 28

Affected municipalities:
  With some missing data: 38 (0.47%)
  Complete (no missing): 8094 (99.53%)

Data integrity (among affected municipalities):
  Minimum % missing: 3.57%
  Maximum % missing: 96.43%
  Average % missing: 45.96%
  Median % missing: 50.00%

✓ Validation:
  - Total_Years is always 28 for all records
  - Years_With_Data + Years_Missing = 28
  - Pct_Missing range: 0% - 100%


## 8. Export Results

In [12]:
# Export files
print("Exporting results...\n")

DATA_DIR_DER_NAN = Path(
    r"/workspaces/rural-migration-land-use-spain/data/demography/derived/nan"
)

# 1. Detailed format (long)
df_detailed.to_csv(DATA_DIR_DER_NAN / "nan_analysis_detailed.csv", index=False, encoding='utf-8')
print(f"✓ nan_analysis_detailed.csv")
print(f"  - Rows: {len(df_detailed)} (one per category)")
print(f"  - Unique municipalities: {df_detailed['Mun_Code'].nunique()}")

# 2. Pivoted format (wide)
df_pivoted.to_csv(DATA_DIR_DER_NAN / "nan_analysis_pivoted.csv", index=False, encoding='utf-8')
print(f"\n✓ nan_analysis_pivoted.csv")
print(f"  - Rows: {len(df_pivoted)} (one per municipality)")
print(f"  - Columns: {len(df_pivoted.columns)}")

# 3. Summary by county
df_comarca.to_csv(DATA_DIR_DER_NAN / "nan_analysis_by_comarca.csv", index=False, encoding='utf-8')
print(f"\n✓ nan_analysis_by_comarca.csv")
print(f"  - Rows: {len(df_comarca)} (counties with affected municipalities)")

print(f"\n✓ Export completed successfully")

Exporting results...

✓ nan_analysis_detailed.csv
  - Rows: 114 (one per category)
  - Unique municipalities: 38

✓ nan_analysis_pivoted.csv
  - Rows: 38 (one per municipality)
  - Columns: 23

✓ nan_analysis_by_comarca.csv
  - Rows: 33 (counties with affected municipalities)

✓ Export completed successfully


## 9. Geographic Enrichment with Municipal Boundaries

In [13]:
# Load geographic data as GeoDataFrame
import geopandas as gpd
from pathlib import Path

print("Loading geographic data for diagnosis...\n")

# Path to geographic GPKG
gpkg_geo = Path(
    r"/workspaces/rural-migration-land-use-spain/data/spatial/derived/mun_geographic_administrative_hierarchy.gpkg"
)

# Load geographic data
gdf_geo = gpd.read_file(gpkg_geo)

# Ensure Mun_Code is string with 5 digits (should already be, but verify)
gdf_geo['Mun_Code'] = gdf_geo['Mun_Code'].astype(str).str.zfill(5)

print(f"✓ Geographic data loaded")
print(f"  - Records: {len(gdf_geo)}")
print(f"  - Mun_Code format: {gdf_geo['Mun_Code'].dtype} (e.g., {gdf_geo['Mun_Code'].iloc[0]})")
print(f"  - Geometry type: {gdf_geo.geometry.type.unique()}")
print(f"  - CRS: {gdf_geo.crs}\n")

Loading geographic data for diagnosis...

✓ Geographic data loaded
  - Records: 8132
  - Mun_Code format: str (e.g., 09298)
  - Geometry type: <StringArray>
['MultiPolygon']
Length: 1, dtype: str
  - CRS: EPSG:4258



In [14]:
# ============================================
# CHECK MUN_CODE FORMAT
# ============================================
print("\n" + "="*70)
print("MUN_CODE FORMAT VERIFICATION")
print("="*70)

# IMPORTANT: Ensure df_pivoted Mun_Code is string with 5 digits
df_pivoted['Mun_Code'] = df_pivoted['Mun_Code'].astype(str).str.zfill(5)
df_detailed['Mun_Code'] = df_detailed['Mun_Code'].astype(str).str.zfill(5)

# Check df_pivoted codes
print("\ndf_pivoted Mun_Code analysis:")
print(f"  - Data type: {df_pivoted['Mun_Code'].dtype}")
print(f"  - Length range: {df_pivoted['Mun_Code'].str.len().min()} to {df_pivoted['Mun_Code'].str.len().max()}")
print(f"  - Sample codes: {df_pivoted['Mun_Code'].head(5).tolist()}")

short_codes = df_pivoted[df_pivoted['Mun_Code'].str.len() < 5]
if len(short_codes) > 0:
    print(f"  - Codes with length < 5:")
    for idx, row in short_codes.iterrows():
        print(f"    {row['Mun_Code']} | {row['Mun_Name']} | {row['Prov_Name']}")
else:
    print(f"  - ✓ All codes have 5 digits")

# Check gdf_geo codes
print("\ngdf_geo Mun_Code analysis:")
print(f"  - Data type: {gdf_geo['Mun_Code'].dtype}")
print(f"  - Length range: {gdf_geo['Mun_Code'].str.len().min()} to {gdf_geo['Mun_Code'].str.len().max()}")
print(f"  - Sample codes: {gdf_geo['Mun_Code'].head(5).tolist()}")

short_codes_geo = gdf_geo[gdf_geo['Mun_Code'].str.len() < 5]
if len(short_codes_geo) > 0:
    print(f"  - Codes with length < 5: Found {len(short_codes_geo)} codes")
else:
    print(f"  - ✓ All codes have 5 digits")


MUN_CODE FORMAT VERIFICATION

df_pivoted Mun_Code analysis:
  - Data type: str
  - Length range: 5 to 5
  - Sample codes: ['06903', '18077', '18106', '25913', '21902']
  - ✓ All codes have 5 digits

gdf_geo Mun_Code analysis:
  - Data type: str
  - Length range: 5 to 5
  - Sample codes: ['09298', '09301', '09302', '05105', '05106']
  - ✓ All codes have 5 digits


In [15]:
# ============================================
# DIAGNOSTIC: Check data alignment before merge
# ============================================
print("="*70)
print("DIAGNOSTIC: Data alignment verification")
print("="*70)

print("\n=== CENSUS DATA WITH NaN ===")
print(f"Mun_Code dtype: {df_pivoted['Mun_Code'].dtype}")
print(f"Sample codes: {df_pivoted['Mun_Code'].head(10).tolist()}")
print(f"Total municipalities: {len(df_pivoted)}")

print("\n=== GEOGRAPHIC DATA ===")
print(f"Mun_Code dtype: {gdf_geo['Mun_Code'].dtype}")
print(f"Sample codes: {gdf_geo['Mun_Code'].head(10).tolist()}")
print(f"Total municipalities: {len(gdf_geo)}")

# Both should now be strings with 5 digits
censo_codes = set(df_pivoted['Mun_Code'])
geo_codes = set(gdf_geo['Mun_Code'])

print("\n=== CODE MATCHING ===")
missing_in_geo = censo_codes - geo_codes
common_codes = censo_codes & geo_codes

if missing_in_geo:
    print(f"⚠ Codes in census but NOT in geographic data: {len(missing_in_geo)}")
    print(f"  Codes: {sorted(list(missing_in_geo))[:20]}")
else:
    print(f"✓ All census codes found in geographic data")

print(f"\n✓ Common codes: {len(common_codes)}/{len(censo_codes)}")

# Verify formats match
if df_pivoted['Mun_Code'].dtype == gdf_geo['Mun_Code'].dtype:
    print(f"\n✓ Both datasets have matching Mun_Code dtype: {df_pivoted['Mun_Code'].dtype}")
else:
    print(f"\n⚠ Mismatch in Mun_Code dtype: census={df_pivoted['Mun_Code'].dtype}, geo={gdf_geo['Mun_Code'].dtype}")

DIAGNOSTIC: Data alignment verification

=== CENSUS DATA WITH NaN ===
Mun_Code dtype: str
Sample codes: ['06903', '18077', '18106', '25913', '21902', '06902', '08905', '19171', '14902', '04904']
Total municipalities: 38

=== GEOGRAPHIC DATA ===
Mun_Code dtype: str
Sample codes: ['09298', '09301', '09302', '05105', '05106', '05107', '05108', '05109', '05114', '05116']
Total municipalities: 8132

=== CODE MATCHING ===
✓ All census codes found in geographic data

✓ Common codes: 38/38

✓ Both datasets have matching Mun_Code dtype: str


In [16]:
# SHOW MISSING MUNICIPALITIES DETAILS
if missing_in_geo:
    print("\n" + "="*70)
    print("MUNICIPALITIES WITH MISSING GEOMETRY")
    print("="*70 + "\n")
    
    missing_details = df_pivoted[df_pivoted['Mun_Code'].isin(missing_in_geo)].copy()
    
    for idx, row in missing_details.iterrows():
        print(f"  {row['Mun_Code']} | {row['Mun_Name']} | {row['Comarca_Name']} | {row['Prov_Name']} | {row['CCAA_Name']}")
else:
    print("\n✓ All municipalities have matching geometry")


✓ All municipalities have matching geometry


In [17]:
# Enrich statistical data with geographic boundaries
print("\nEnriching statistical data with geographic boundaries...\n")

# Select only necessary columns from geographic data
geo_cols = ['Mun_Code', 'geometry']
gdf_geo_select = gdf_geo[geo_cols].copy()

# 1. Join detailed with geometry
gdf_detailed_geo = gpd.GeoDataFrame(
    df_detailed.merge(gdf_geo_select, on='Mun_Code', how='left'),
    geometry='geometry',
    crs=gdf_geo.crs
)

print(f"✓ Detailed data enriched with geometry")
print(f"  - Records: {len(gdf_detailed_geo)}")
print(f"  - Geometry match: {gdf_detailed_geo.geometry.notna().sum()}/{len(gdf_detailed_geo)}")

# 2. Join pivoted with geometry
gdf_pivoted_geo = gpd.GeoDataFrame(
    df_pivoted.merge(gdf_geo_select, on='Mun_Code', how='left'),
    geometry='geometry',
    crs=gdf_geo.crs
)

print(f"\n✓ Pivoted data enriched with geometry")
print(f"  - Records: {len(gdf_pivoted_geo)}")
print(f"  - Geometry match: {gdf_pivoted_geo.geometry.notna().sum()}/{len(gdf_pivoted_geo)}")

# Show which municipalities are missing geometry
missing_geom = gdf_pivoted_geo[gdf_pivoted_geo.geometry.isna()]
if len(missing_geom) > 0:
    print(f"\n⚠ Municipalities WITHOUT geometry:")
    for idx, row in missing_geom.iterrows():
        print(f"  {row['Mun_Code']} | {row['Mun_Name']} | {row['Comarca_Name']} | {row['Prov_Name']} | {row['CCAA_Name']}")

# 3. Join comarca with geometry (aggregate geometry per comarca)
gdf_geo_by_comarca = gdf_geo.dissolve(by='Comarca_Code', aggfunc='first')
gdf_comarca_geo = gpd.GeoDataFrame(
    df_comarca.set_index('Comarca_Code').join(
        gdf_geo_by_comarca[['geometry']],
        how='left'
    ),
    geometry='geometry',
    crs=gdf_geo.crs
).reset_index()

print(f"\n✓ Comarca data enriched with aggregated geometry")
print(f"  - Records: {len(gdf_comarca_geo)}")
print(f"  - Geometry match: {gdf_comarca_geo.geometry.notna().sum()}/{len(gdf_comarca_geo)}")

print(f"\n✓ Geographic enrichment completed")


Enriching statistical data with geographic boundaries...

✓ Detailed data enriched with geometry
  - Records: 114
  - Geometry match: 114/114

✓ Pivoted data enriched with geometry
  - Records: 38
  - Geometry match: 38/38

✓ Comarca data enriched with aggregated geometry
  - Records: 33
  - Geometry match: 33/33

✓ Geographic enrichment completed


In [18]:
# Export all GeoDataFrames to a single GeoPackage with multiple layers
gpkg_path = DATA_DIR_DER_NAN / "nan_analysis_geo.gpkg"

# Layer 1: Detailed
gdf_detailed_geo.to_file(gpkg_path, driver='GPKG', layer='nan_detailed')
# 114 registros (38 municipios × 3 categorías)

# Layer 2: Pivoted
gdf_pivoted_geo.to_file(gpkg_path, driver='GPKG', layer='nan_pivoted')
# 38 registros (1 municipio por fila)

# Layer 3: Comarca
gdf_comarca_geo.to_file(gpkg_path, driver='GPKG', layer='nan_comarca')
# 33 registros (1 comarca con geometría agregada)

## 10. Conclusions and Recommendations

In [19]:
print("="*70)
print("CONCLUSIONS AND RECOMMENDATIONS")
print("="*70)

print(f"\n✓ DATASET QUALITY")
print(f"  • The dataset is VERY CLEAN: only {(len(df_pivoted)/padron['Mun_Code'].nunique()*100):.2f}% of municipalities affected")
print(f"  • No municipality has completely missing data")
print(f"  • Missing values are concentrated in older years (1996-2015)")
print(f"  • From 2019 onwards: practically no missing values")

print(f"\n✓ USAGE RECOMMENDATIONS")
print(f"  1. For temporal analysis: use period 2010-2024 for maximum reliability")
print(f"  2. For 1996-2024 analysis: document municipalities with NaN in reports")
print(f"  3. For regional analysis: safe to use without restrictions")
print(f"  4. For exploratory analysis: NaN is not a problem")

print(f"\n✓ NEXT STEPS")
print(f"  1. Review generated CSV files")
print(f"  2. Consider value imputation for complex analysis")
print(f"  3. Use the GPKG for GIS integration (QGIS/ArcGIS)")
print(f"  4. Consult metadata for details on each municipality")

CONCLUSIONS AND RECOMMENDATIONS

✓ DATASET QUALITY
  • The dataset is VERY CLEAN: only 0.47% of municipalities affected
  • No municipality has completely missing data
  • Missing values are concentrated in older years (1996-2015)
  • From 2019 onwards: practically no missing values

✓ USAGE RECOMMENDATIONS
  1. For temporal analysis: use period 2010-2024 for maximum reliability
  2. For 1996-2024 analysis: document municipalities with NaN in reports
  3. For regional analysis: safe to use without restrictions
  4. For exploratory analysis: NaN is not a problem

✓ NEXT STEPS
  1. Review generated CSV files
  2. Consider value imputation for complex analysis
  3. Use the GPKG for GIS integration (QGIS/ArcGIS)
  4. Consult metadata for details on each municipality
